# Lab 04: Spark Streaming

**Môn học:** Nhập môn Dữ liệu lớn - Nhóm 4  
**Repository lựa chọn:** `huggingface/peft`

## 0. Chuẩn bị notebook

Các file evidence được đặt trong thư mục:

```text
evidence/
output/
configs/
jobs/
```

Cell dưới đây dò tìm project root có chứa `evidence/`

In [63]:
from pathlib import Path
import json
from pprint import pprint


def find_project_root(start: Path = Path.cwd()) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "evidence").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
EVIDENCE_DIR = PROJECT_ROOT / "evidence"
OUTPUT_DIR = PROJECT_ROOT / "output"
CONFIG_DIR = PROJECT_ROOT / "configs"
JOBS_DIR = PROJECT_ROOT / "jobs"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("EVIDENCE_DIR =", EVIDENCE_DIR)

PROJECT_ROOT = /Users/hvpu/Downloads/group4
EVIDENCE_DIR = /Users/hvpu/Downloads/group4/evidence


In [ ]:
# Một số hàm hỗ trợ 

def read_text(path, max_chars=None):
    path = Path(path)
    if not path.exists():
        return f"[MISSING] {path}"
    text = path.read_text(encoding="utf-8", errors="replace")
    if max_chars is not None and len(text) > max_chars:
        return text[:max_chars] + f"\n... [truncated, total {len(text)} chars]"
    return text

def read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"[MISSING] {path}")
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def show_file(path, max_chars=4000):
    print(read_text(path, max_chars=max_chars))

## Task 1. Repository Cloning and File Discovery

Điểm bắt đầu của pipeline là chọn một repository. Repo nhóm chọn là `huggingface/peft`.

Repository được shallow clone để tải nhanh:

```bash
git clone --depth 1 https://github.com/huggingface/peft.git peft
```

Commit được sử dụng: `79f4c362248d3b3b4bc2ed24704ed3183528c53f`

Sau khi clone, nhóm chạy discovery theo hai chế độ: có tính test files và không tính test files. Các thư mục cache/runtime như `__pycache__`, `.pytest_cache`, `output`, `checkpoints` cũng được loại khỏi discovery để tránh đưa artifact sinh tự động vào xử lý.


#### Lệnh chạy Task 1

```bash
PYTHONPATH=src python3 -m group4_lab.cli discover --repo peft --include-tests
```


In [77]:
discover_no_tests = read_json(EVIDENCE_DIR / "discover_no_tests.json")
discover_include_tests = read_json(EVIDENCE_DIR / "discover_include_tests.json")

print("Discovery không tính tests:")
pprint(discover_no_tests)

print("\nDiscovery có tính tests:")
pprint(discover_include_tests)

Discovery không tính tests:
{'by_top_level_dir': {'docs': 1,
                      'examples': 93,
                      'method_comparison': 17,
                      'scripts': 8,
                      'setup.py': 1,
                      'src': 245},
 'repo_root': '/Users/hvpu/Downloads/group4/peft',
 'sample_files': ['docs/source/_config.py',
                  'examples/KappaTune/experiments_kappatune_peft.py',
                  'examples/adamss_finetuning/glue_adamss_asa_example.py',
                  'examples/adamss_finetuning/glue_adamss_asa_manual_example.py',
                  'examples/adamss_finetuning/image_classification_adamss_asa.py',
                  'examples/alora_finetuning/alora_finetuning.py',
                  'examples/arrow_multitask/arrow_phi3_mini.py',
                  'examples/bdlora_finetuning/chat.py',
                  'examples/beft_finetuning/beft_finetuning.py',
                  'examples/boft_controlnet/__init__.py'],
 'total_files': 365}

Discove

### Reflection Task 1

Discovery cho thấy:

- Không tính tests: 365 file Python
- Có tính tests: 431 file Python

Nhóm chọn tập 365 file Python không tính test files cho các bước parse và sinh event vì như vậy giúp pipeline tập trung vào source code chính, thay vì để test cases làm graph lớn và nhiễu hơn.

Điểm nhóm phải cân nhắc là có nên loại thêm setup files hay không. Vì chỉ có một `setup.py`, không gây ảnh hưởng quá nhiều đến graph, nhóm giữ lại file này và chỉ loại tests cùng các thư mục sinh tự động.

## Task 2. Incremental CPG Parser Service

Bước tiếp theo là biến source code thành event. Nhóm xây parser theo hướng incremental: mỗi file Python được parse độc lập và sinh ra node, edge, metadata hoặc error event. Cách này giúp pipeline có thể replay riêng một file khi file đó thay đổi, thay vì phải xử lý lại cả repository.

Các loại event chính:

| Loại event | Ý nghĩa |
|---|---|
| `node` | Biểu diễn node trong AST/CPG |
| `edge` | Biểu diễn quan hệ giữa các node |
| `metadata` | Thông tin tổng hợp của từng file |
| `error` | Lỗi parser nếu có |

Parser trích xuất AST nodes/edges, CFG edges, DFG edges và call edges. Stable ID được tạo từ repo, file, scope và fingerprint của AST node để hỗ trợ replay/idempotency ở các task sau.


#### Lệnh chạy Task 2

```bash
# Parse toàn bộ PEFT offline và ghi event ra JSONL.

mkdir -p output

PYTHONPATH=src python3 -m group4_lab.cli parse-repo \
  --repo peft \
  --repo-name huggingface/peft \
  --commit-sha 79f4c362248d3b3b4bc2ed24704ed3183528c53f \
  --publisher console \
  --publisher-output output/peft-events.jsonl
```


In [80]:
parse_summary = read_json(EVIDENCE_DIR / "parse_summary.json")
pprint(parse_summary)

{'edge_type_counts': {'AST_CHILD': 246361,
                      'CALLS': 23036,
                      'CFG_ELSE': 3,
                      'CFG_EXCEPT': 61,
                      'CFG_FALSE': 783,
                      'CFG_FINALLY': 6,
                      'CFG_LOOP': 2630,
                      'CFG_NEXT': 20179,
                      'CFG_TRUE': 4169,
                      'CFG_TRY': 138,
                      'DFG': 55961},
 'event_file': 'output/peft-events.jsonl',
 'event_lines': 604529,
 'sample_topics_written_to': 'evidence/sample_events_by_topic.json',
 'top_node_kinds': {'Assign': 15346,
                    'Attribute': 17148,
                    'BinOp': 2780,
                    'Call': 23018,
                    'Compare': 4118,
                    'Constant': 49591,
                    'Dict': 1540,
                    'Expr': 6692,
                    'FormattedValue': 2289,
                    'FunctionDef': 2528,
                    'If': 5841,
                    'I

### Reflection Task 2

Parser xử lý toàn bộ tập 365 file Python và sinh ra:

- 250,837 node events
- 353,327 edge events
- 365 metadata events
- 0 errors

Số parser errors là 0, cho thấy parser đủ ổn định để làm producer cho các bước Kafka, Neo4j và MongoDB phía sau.

Phần chạy tốt nhất ở task này là parser có thể đi qua toàn bộ source chính của PEFT mà không lỗi syntax/parser. Điểm khó nằm ở thao tác replay: nếu mỗi lần parse lại sinh ID khác nhau thì downstream sẽ rất dễ bị trùng dữ liệu. Nhóm xử lý bằng stable hashing dựa trên repo, file, scope và fingerprint của node, thay vì dùng ID ngẫu nhiên theo mỗi lần chạy.

## Task 3. Kafka Topic Design

Khi parser đã sinh event, Kafka đóng vai trò như "trạm trung chuyển". Nhóm không gom tất cả event vào một topic chung, vì graph events và metadata events sẽ đi tới các sink khác nhau. Thay vào đó, event được chia theo loại dữ liệu để downstream consume đúng phần mình cần.

| Kafka topic | Vai trò trong pipeline | Downstream chính |
|---|---|---|
| `peft.cpg.nodes` | Chứa node events của Code Property Graph | Neo4j |
| `peft.cpg.edges` | Chứa edge events của Code Property Graph | Neo4j |
| `peft.source.metadata` | Chứa metadata cấp file: hash, số dòng, số node/edge, số hàm/class/import | Spark Structured Streaming -> MongoDB |
| `peft.parser.errors` | Ghi nhận lỗi parse để pipeline không dừng toàn bộ batch | Monitoring/debug |

![Kafka pipeline overview](figures/kafka_pipeline.svg)

Các trường dữ liệu chính của event gồm:

```text
schema_version
event_time
repo
commit_sha
file_path
element_id / edge_id / content_hash
properties
```

Mỗi Kafka event có một key ổn định. Node dùng `element_id`, edge dùng `edge_id`, metadata dùng `file_path`. Neo4j connector dùng key/ID này với `MERGE` để chống duplicate cho node và edge. Với MongoDB, Spark sink dùng `repo,file_path` làm key upsert, cấu hình `operationType=replace` và `upsertDocument=true`, nên khi replay cùng một source file thì metadata document được thay thế bằng phiên bản mới thay vì append thành bản ghi trùng.

#### Lệnh chạy Task 3

```bash
# Khởi động Kafka, tạo topic và publish sample event từ loraplus.py.

docker compose up -d zookeeper kafka

python3 scripts/create_topics.py

docker compose exec kafka kafka-topics --create --topic peft.cpg.nodes --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
docker compose exec kafka kafka-topics --create --topic peft.cpg.edges --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
docker compose exec kafka kafka-topics --create --topic peft.source.metadata --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
docker compose exec kafka kafka-topics --create --topic peft.parser.errors --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1

PYTHONPATH=src python3 -m group4_lab.cli parse-file \
  --file peft/src/peft/optimizers/loraplus.py \
  --repo-root peft \
  --repo-name huggingface/peft \
  --commit-sha 79f4c362248d3b3b4bc2ed24704ed3183528c53f \
  --publisher kafka \
  --bootstrap-servers localhost:9092
```


In [ ]:
# các lệnh topic kafka
print("Kafka topic commands:")
show_file(EVIDENCE_DIR / "kafka_topic_commands.txt", max_chars=4000)

Kafka topic commands:
kafka-topics --create --topic peft.cpg.nodes --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
kafka-topics --create --topic peft.cpg.edges --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
kafka-topics --create --topic peft.source.metadata --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
kafka-topics --create --topic peft.parser.errors --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1



In [68]:
# trạng thái topic kafka
print("Kafka topics describe:")
show_file(EVIDENCE_DIR / "kafka_topics_describe.txt", max_chars=6000)

Kafka topics describe:
Topic: peft.cpg.nodes	TopicId: o6KUKi7oSe21UB66IHghkQ	PartitionCount: 6	ReplicationFactor: 1	Configs: 
	Topic: peft.cpg.nodes	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 2	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 3	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 4	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 5	Leader: 1	Replicas: 1	Isr: 1
Topic: peft.cpg.edges	TopicId: 4YYB47O0TYSYcaWctvliHw	PartitionCount: 6	ReplicationFactor: 1	Configs: 
	Topic: peft.cpg.edges	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 2	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 3	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 4	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 5	Leader: 1	Replicas: 

#### Một số event sample từ full parse (offline):

In [69]:
sample_events = read_json(EVIDENCE_DIR / "sample_events_by_topic.json")
pprint(sample_events)

{'peft.cpg.edges': {'key': '2bafb589ec0db60d6e90f9110e241eedd61df446',
                    'topic': 'peft.cpg.edges',
                    'value': {'commit_sha': '79f4c362248d3b3b4bc2ed24704ed3183528c53f',
                              'edge_id': '2bafb589ec0db60d6e90f9110e241eedd61df446',
                              'edge_type': 'AST_CHILD',
                              'event_time': '2026-07-13T03:58:52.833413Z',
                              'file_path': '/Users/hvpu/Downloads/group4/peft/docs/source/_config.py',
                              'properties': {},
                              'repo': 'huggingface/peft',
                              'schema_version': 1,
                              'source_id': 'f289069d7149c9642c7fc928087221017d8db69a',
                              'target_id': 'd68235b762b56a51aa30be942f1e8702d2327c48'}},
 'peft.cpg.nodes': {'key': 'f289069d7149c9642c7fc928087221017d8db69a',
                    'topic': 'peft.cpg.nodes',
                    'val

### Reflection Task 3

Các Kafka topic đã được tạo trong Docker Kafka, sau đó parser publish event mẫu vào đúng topic tương ứng. Với Kafka live sample từ `loraplus.py`, parser sinh được:

```text
peft.cpg.nodes: 233
peft.cpg.edges: 321
peft.source.metadata: 1
```

Thiết kế topic tách riêng node, edge, metadata và parser errors giúp pipeline dễ đọc hơn: Neo4j chỉ cần consume graph events, còn Spark/MongoDB chỉ cần metadata events.

## Task 4. Graph Topology Ingestion into Neo4j

Sau khi Kafka đã có graph events, phần Neo4j là nơi kiểm tra xem node/edge có thể đi vào graph database thật hay không. Nhóm dùng Neo4j Kafka Connector Sink.

Nhóm tách node và edge thành hai connector:

```text
configs/neo4j/node-sink-connector.json
configs/neo4j/edge-sink-connector.json
```

Connector sử dụng class:

```text
org.neo4j.connectors.kafka.sink.Neo4jConnector
```

Neo4j được cấu hình constraint theo ID để hỗ trợ `MERGE`. Nhờ vậy, khi cùng một event được publish lại, Neo4j cập nhật node/edge cũ thay vì tạo bản ghi trùng.

#### Lệnh chạy Task 4

```bash
# Khởi động Neo4j/Kafka Connect, apply constraint và tạo Neo4j Sink connectors.

docker compose up -d zookeeper kafka neo4j connect

cat configs/neo4j/constraints.cypher | docker compose exec -T neo4j cypher-shell -u neo4j -p password123

curl -X POST http://localhost:8083/connectors \
  -H "Content-Type: application/json" \
  --data @configs/neo4j/node-sink-connector.json

curl -X POST http://localhost:8083/connectors \
  -H "Content-Type: application/json" \
  --data @configs/neo4j/edge-sink-connector.json
```


Nhóm lưu raw evidence trong các file sau:

```text
evidence/replay_end_to_end/connect_plugins_final.json
evidence/replay_end_to_end/node_connector_status_final.json
evidence/replay_end_to_end/edge_connector_status_final.json
evidence/neo4j_constraints.txt
```

Neo4j Kafka Connector plugin được load, node/edge sink connectors ở trạng thái `RUNNING`, và Neo4j constraints đã được apply để hỗ trợ `MERGE` theo ID. Kết quả Neo4j được kiểm tra bằng Neo4j Browser. Hai ảnh đầu tiên cho thấy graph events đã đi vào Neo4j và có thể query được số lượng node/relationship sau khi Kafka Connect consume dữ liệu.

![Neo4j node count query](assets/neo4j-node-count.jpg)

![Neo4j relationship count query](assets/neo4j-relationship-count.jpg)

Ngoài việc query count, nhóm cũng thử mở full graph trực tiếp trong Neo4j Browser để quan sát topology. Phần này có giới hạn thực tế: Neo4j Browser giới hạn số node hiển thị khoảng 20,000-30000 node, trong khi full graph của PEFT lớn hơn rất nhiều. Khi cố hiển thị quá nhiều node/relationship, máy local có lúc bị quá tải RAM/CPU, trình duyệt bị crash hoặc giao diện Neo4j tự ngắt phản hồi. Ảnh dưới đây là hiển thị graph tại node limit 20000.

![Neo4j graph limit](assets/neo4j_full_graph_limit.jpeg)

### Reflection Task 4

Phần Task 4 chủ yếu kiểm chứng việc nối Kafka Connect với Neo4j: connector plugin được load, node/edge sink connector tạo thành công, constraint trong Neo4j được apply, và graph counts trong Neo4j có thay đổi sau khi connector consume event. Giới hạn của phần này là nhóm chưa ingest live toàn bộ full graph 365 file vào Neo4j local. Full graph đã được parse offline với hơn 604k graph events, còn phần Neo4j connector được kiểm chứng bằng graph events của một file sample để giữ môi trường Docker/Neo4j ổn định.


## Task 5. Source Metadata Ingestion into MongoDB

Nếu Neo4j là nơi lưu graph topology, thì MongoDB được dùng để lưu metadata cấp file. Metadata khá nhẹ so với graph, nên nhóm chạy full 365 files Python qua luồng:
```text
Kafka topic peft.source.metadata -> Spark Structured Streaming -> MongoDB collection
```

#### Lệnh chạy Task 5

```bash
docker compose up -d zookeeper kafka mongo

# Nếu output/peft-events.jsonl chưa có, chạy lại Task 2 trước.
# Sau đó publish các metadata events vào topic peft.source.metadata.

JAVA_HOME=/tmp/group4-jdk17/Contents/Home \
GROUP4_SPARK_CHECKPOINT=checkpoints/full_metadata_stream \
GROUP4_MONGO_COLLECTION=peft_metadata_full \
spark-submit \
  --conf spark.jars.ivy=/tmp/group4-ivy \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.3,org.mongodb.spark:mongo-spark-connector_2.13:10.5.0 \
  jobs/mongo_streaming_available_now.py
```


### Metadata sample

Ảnh dưới đây là sample document xem bằng MongoDB Compass. Metadata từ Kafka/Spark đã được ghi vào MongoDB, đồng thời thể hiện rõ schema và các field sau khi streaming.

![MongoDB Compass metadata document](assets/mongo-compass-peft-metadata.jpg)


In [72]:
print("Mongo metadata sample:")
show_file(EVIDENCE_DIR / "mongo_metadata_sample.txt", max_chars=4000)

print("\nSpark Mongo metadata sample:")
show_file(EVIDENCE_DIR / "spark_mongo_metadata_sample.txt", max_chars=4000)

Mongo metadata sample:
[
  {
    file_path: 'peft/src/peft/optimizers/loraplus.py',
    ast_node_count: 233,
    ast_edge_count: 227,
    cfg_edge_count: 25,
    dfg_edge_count: 50,
    call_edge_count: 19
  }
]


Spark Mongo metadata sample:
[
  {
    file_path: 'peft/src/peft/optimizers/loraplus.py',
    ast_node_count: Long('233'),
    ast_edge_count: Long('227'),
    cfg_edge_count: Long('25'),
    dfg_edge_count: Long('50'),
    call_edge_count: Long('19')
  }
]



### Checkpoint

Spark Structured Streaming được cấu hình với `checkpointLocation=checkpoints/full_metadata_stream`. Thư mục checkpoint này lưu trạng thái của streaming query, bao gồm metadata của query, Kafka offsets đã xử lý và commit log của từng micro-batch. Nhờ đó, khi job chạy lại, Spark có thể tiếp tục từ trạng thái đã lưu thay vì đọc lại toàn bộ Kafka topic từ đầu.

Sau khi job hoàn tất, các file checkpoint như `metadata`, `offsets/0`, `commits/0` và `sources/0/0` được ghi nhận trong `evidence/full_metadata/spark_full_metadata_checkpoint_files.txt`.


### Ingest full metadata

Tiến hành extract toàn bộ metadata từ `output/peft-events.jsonl`, nhóm thu được kết quả:

```text
365 metadata events
365 unique file paths
```

Sau đó publish vào Kafka topic `peft.source.metadata` và chạy Spark Structured Streaming `availableNow` để ghi vào MongoDB.

In [73]:
full_metadata_dir = EVIDENCE_DIR / "full_metadata"

print("Full metadata extract summary:")
pprint(read_json(full_metadata_dir / "full_metadata_extract_summary.json"))

print("\nKafka offsets sau khi publish 365 metadata events:")
show_file(full_metadata_dir / "kafka_full_metadata_offsets.txt", max_chars=4000)

print("\nSpark full metadata log:")
show_file(full_metadata_dir / "spark_full_metadata.log", max_chars=6000)

print("\nMongo full metadata count + sample:")
show_file(full_metadata_dir / "mongo_full_metadata_count_sample.json", max_chars=6000)

print("\nSpark checkpoint files:")
show_file(full_metadata_dir / "spark_full_metadata_checkpoint_files.txt", max_chars=4000)

Full metadata extract summary:
{'metadata_events': 365,
 'nul_bytes': 0,
 'source': 'output/peft-events.jsonl',
 'target': 'evidence/full_metadata/peft.source.metadata.full.jsonl',
 'unique_file_paths': 365}

Kafka offsets sau khi publish 365 metadata events:
peft.source.metadata:0:67
peft.source.metadata:1:180
peft.source.metadata:2:118


Spark full metadata log:
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/13 12:01:50 WARN Utils: Your hostname, Uyens-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.81 instead (on interface en0)
26/07/13 12:01:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/hvpu/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /tmp/group4-ivy/cache
The jars for the packages stored in: /tmp/group4-ivy/jars
org.apache.spark#spark-sql-kafka

### Reflection Task 5

Kết quả full metadata live:

```text
Kafka offsets:
partition 0: 67
partition 1: 180
partition 2: 118
Tổng: 365

Spark:
numInputRows: 365
numOutputRows: 365

MongoDB:
collection lab4.peft_metadata_full có 365 documents
```

Như vậy, toàn bộ metadata của 365 file Python đã được stream từ Kafka sang MongoDB thông qua Spark Structured Streaming.

![MongoDB Compass collection overview](assets/mongo-compass-peft-collection-overview.png)

## Task 6. Idempotent Replay Verification

Nhóm dùng file `peft/src/peft/optimizers/loraplus.py` để tạo hai phiên bản: bản gốc và bản sau chỉnh sửa.

| Chỉ số parser replay | Giá trị |
|---|---:|
| Before events | 470 |
| After events | 512 |
| Stable IDs | 444 |
| Added | 68 |
| Removed | 26 |

Kết quả cho thấy phần lớn ID vẫn ổn định giữa hai phiên bản, đồng thời parser vẫn phát hiện được event mới và event bị loại bỏ khi file thay đổi.

#### Lệnh chạy Task 6

```bash
PYTHONPATH=src python3 -m group4_lab.cli replay-file \
  --before-file evidence/replay/loraplus_before.py \
  --after-file evidence/replay/loraplus_after.py \
  --repo-root peft \
  --repo-name huggingface/peft \
  --commit-sha 79f4c362248d3b3b4bc2ed24704ed3183528c53f
```


Raw evidence của bước parser replay được lưu trong `evidence/replay/`, gồm hai phiên bản `loraplus_before.py`, `loraplus_after.py`, event JSONL của bản sau chỉnh sửa và file `loraplus_replay_summary.json`.


### Replay graph trong Neo4j

Neo4j được dùng để kiểm tra idempotency của graph store ở mức chống duplicate khi các event được publish lại. Khi publish lại phiên bản gốc của `loraplus.py`, số lượng node và edge không thay đổi, cho thấy cơ chế `MERGE` cùng với constraint theo ID hoạt động đúng đối với các event trùng lặp. Khi publish phiên bản đã chỉnh sửa, số lượng node và edge tăng lên tương ứng với nội dung mới của file.

| Lần kiểm tra | Nodes | Edges |
|---|---:|---:|
| Phiên bản gốc của `loraplus.py` | 177 | 292 |
| Phiên bản gốc, publish lại lần 2 | 177 | 292 |
| Phiên bản đã chỉnh sửa của `loraplus.py` | 199 | 338 |

![Neo4j task 6](assets/task6_neo4j.jpg)


Raw query output cho ba lần kiểm tra Neo4j được lưu tại:

```text
evidence/replay_end_to_end/neo4j_before_graph_connector.txt
evidence/replay_end_to_end/neo4j_after_republish_before_connector.txt
evidence/replay_end_to_end/neo4j_after_replay_connector.txt
```

Bảng và ảnh phía trên là phần tóm tắt từ các file này. Điểm chính cần đọc là: publish lại cùng bản gốc giữ nguyên count, còn bản sau chỉnh sửa làm count thay đổi.


### Replay metadata trong MongoDB

MongoDB được sử dụng để kiểm tra metadata của cùng một source file trước và sau khi chỉnh sửa. Evidence cho thấy sau repaly collection có hai document cùng `file_path` nhưng khác `content_hash`: một document cho bản gốc và một document cho bản sau chỉnh sửa.

Với cấu hình `operationType=replace`, `upsertDocument=true` và key `repo,file_path`, kết quả này chứng minh metadata của cùng source file được ghi theo cơ chế upsert: replay phiên bản mới thay thế document logic cũ thay vì append thành bản ghi trùng.

| Chỉ số | Bản gốc | Bản sau chỉnh sửa |
|---|---:|---:|
| AST nodes | 233 | 252 |
| AST edges | 227 | 247 |
| CFG edges | 25 | 26 |
| DFG edges | 50 | 50 |
| Call edges | 19 | 21 |
| Functions | 1 | 2 |

##### Trước khi replay

![MongoDB Compass task 6 before](assets/task6_mongodb_before.jpg)

#### Sau khi replay

![MongoDB Compass task 6 after](assets/task6_mongodb_after.jpg)

Raw evidence cho phần MongoDB/Spark replay được lưu tại:

```text
evidence/replay_end_to_end/mongo_metadata_before_replay.txt
evidence/replay_end_to_end/mongo_metadata_after_replay.txt
evidence/replay_end_to_end/spark_replay_before.log
evidence/replay_end_to_end/spark_replay_after.log
evidence/replay_end_to_end/spark_replay_checkpoint_files.txt
```

Trong notebook, nhóm dùng bảng chỉ số và ảnh MongoDB Compass để trình bày phần dễ đọc hơn. Spark checkpoint có commit `0` và `1`, cho thấy streaming query đã lưu trạng thái xử lý qua hai lượt replay.


### Reflection Task 6

Kết quả replay đã chứng minh được một phần idempotency và incremental behavior. Khi publish lại cùng một phiên bản file, số lượng node và edge trong Neo4j không thay đổi; khi publish phiên bản đã chỉnh sửa, graph và metadata được cập nhật theo nội dung mới. Spark checkpoint của replay đã commit các batch `0` và `1`, cho thấy streaming query lưu và khôi phục trạng thái xử lý giữa các lần chạy.

Hạn chế hiện tại nằm ở Neo4j replay. Connector chỉ sử dụng `MERGE` để chống trùng lặp (upsert) và chưa hỗ trợ delete/tombstone event để xoá 26 removed events khỏi graph.

## 7. Architechture

![Architecture overview](assets/architecture_embedded.png)

### Pipeline

##### **Flow 1: Từ source file đến ParseResult**

```text
PEFT Repository Python Source Files
  |
  v
File Discovery discover_python_files()
  |
  v
CLI group4-lab4 parse-repo
  |
  v
Python CPG Parser
  |
  v
ParseResult: Nodes, Edges, Metadata, Errors
```

**Giải thích:**

1. Repository PEFT là nơi chứa source files.
2. `discover_python_files()` tìm danh sách file `.py` cần xử lý.
3. CLI `parse-repo` chạy parser cho từng file trong danh sách.
4. Parser phân tích code thành CPG data.
5. Kết quả của mỗi file được đóng gói trong `ParseResult`.

##### **Flow 2: Từ ParseResult đến Kafka**

```text
ParseResult
  |
  v
Kafka Publisher
  |
  v
Apache Kafka Topics
```

**Giải thích:**
1. Sau khi parser hoàn thành, kết quả được đóng gói trong `ParseResult`
2. Kafka Publisher chuyển dữ liệu trong `ParseResult` thành các message JSON.
3. Các message được publish lên các Kafka topics tương ứng (`peft.cpg.nodes`, `peft.cpg.edges`, `peft.source.metadata`).
4. Kafka lưu trữ và phân phối các message để các hệ thống downstream (Spark, Kafka Connect, Neo4j) xử lý độc lập.


##### **Flow 3: Metadata Pipeline**

```text
Kafka topic peft.source.metadata
  |
  v
Spark Structured Streaming
  |\
  | \-> Checkpoint
  |
  v
MongoDB lab4.peft_metadata
```

**Giải thích:**

1. Kafka lưu metadata event trong topic `peft.source.metadata`.
2. Spark Structured Streaming subscribe topic này.
3. Spark lấy Kafka `value`, cast sang string, parse JSON theo schema.
4. Spark ghi checkpoint để lưu offset/trạng thái streaming.
5. Spark ghi record metadata vào MongoDB collection `lab4.peft_metadata`.

##### **Flow 4: Graph Database Pipeline**

```text
Kafka topics peft.cpg.nodes + peft.cpg.edges
  |
  v
Kafka Connect
  |
  v
Neo4j
```

**Giải thích:**

1. Kafka lưu node event ở `peft.cpg.nodes`.
2. Kafka lưu edge event ở `peft.cpg.edges`.
3. Kafka Connect chạy Neo4j sink connector.
4. Connector chuyển node/edge event thành node/relationship trong Neo4j.
5. Neo4j cho phép query graph, ví dụ function nào gọi function nào, file nào có bao nhiêu class/function, hoặc node nào liên kết với nhau qua AST/CFG/DFG.

## 8. Kiểm thử và môi trường chạy

Nhóm sử dụng Docker để chạy các service:

| Service | Port |
|---|---|
| Kafka | `localhost:9092` |
| Zookeeper | `localhost:2181` |
| Neo4j Browser | `localhost:7474` |
| Neo4j Bolt | `localhost:7687` |
| MongoDB | `localhost:27017` |

Ngoài ra, nhóm cài `pytest` và chạy unit/manual tests để kiểm tra các phần chính của code.


In [ ]:
print("pytest result:")
show_file(EVIDENCE_DIR / "pytest.txt", max_chars=3000)

print("\nManual tests:")
show_file(EVIDENCE_DIR / "manual_tests.txt", max_chars=3000)

print("\nDocker compose final status:")
show_file(EVIDENCE_DIR / "full_metadata" / "docker_compose_ps_after_full_metadata.txt", max_chars=5000)

pytest result:
....                                                                     [100%]
4 passed in 7.08s


Manual tests:
manual parser tests passed
project_python_files=14
sample_nodes=58
sample_edges=78
sample_replay_stable=135


Docker compose final status:
NAME                 IMAGE                                 COMMAND                  SERVICE     CREATED          STATUS                      PORTS
group4-connect-1     confluentinc/cp-kafka-connect:7.6.1   "bash -lc 'confluent…"   connect     22 minutes ago   Up 22 minutes (unhealthy)   0.0.0.0:8083->8083/tcp, [::]:8083->8083/tcp
group4-kafka-1       confluentinc/cp-kafka:7.6.1           "/etc/confluent/dock…"   kafka       45 minutes ago   Up 45 minutes               0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp
group4-mongo-1       mongo:7.0                             "docker-entrypoint.s…"   mongo       45 minutes ago   Up 45 minutes               0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp
group4-neo4j-1       neo4j

## 9. Cấu trúc source code và config

Các thành phần chính của project:

```text
group4/
├── peft/                         # repository PEFT
├── configs/
│   ├── generated/                # generated configs
│   └── neo4j/                    # Neo4j Kafka Sink connector configs
├── evidence/                     # evidence logs, JSON, query outputs
├── jobs/                         # Spark Structured Streaming jobs
├── output/                       # full event JSONL
├── docker-compose.yml            # Kafka, Neo4j, MongoDB, Kafka Connect
└── docs/book/                    # report
```

Các file evidence quan trọng:

```text
evidence/discover_no_tests.json
evidence/parse_summary.json
evidence/sample_events_by_topic.json
evidence/kafka_topics_describe.txt
evidence/replay_end_to_end/node_connector_status_final.json
evidence/replay_end_to_end/edge_connector_status_final.json
evidence/replay_end_to_end/neo4j_after_replay_connector.txt
evidence/full_metadata/spark_full_metadata.log
evidence/full_metadata/mongo_full_metadata_count_sample.json
```


## 10. Hướng dẫn chạy pipeline

Dưới đây là tổng hợp lệnh để chạy full pipeline:

```bash
# 0. Cài dependencies cần thiết
python3 -m pip install --user -e ".[all]"
python3 -m pip install --user "jupyter-book<2"

# 1. Khởi động hạ tầng Docker
docker compose up -d

# 2. Clone PEFT nếu chưa có
git clone --depth 1 https://github.com/huggingface/peft.git peft

# 3. Discovery repository
PYTHONPATH=src python3 -m group4_lab.cli discover --repo peft
PYTHONPATH=src python3 -m group4_lab.cli discover --repo peft --include-tests

# 4. Parse toàn bộ PEFT offline ra JSONL
mkdir -p output
PYTHONPATH=src python3 -m group4_lab.cli parse-repo \
  --repo peft \
  --repo-name huggingface/peft \
  --commit-sha 79f4c362248d3b3b4bc2ed24704ed3183528c53f \
  --publisher console \
  --publisher-output output/peft-events.jsonl

# 5. Tạo Kafka topics
python3 scripts/create_topics.py

docker compose exec kafka kafka-topics --create --topic peft.cpg.nodes --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
docker compose exec kafka kafka-topics --create --topic peft.cpg.edges --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
docker compose exec kafka kafka-topics --create --topic peft.source.metadata --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
docker compose exec kafka kafka-topics --create --topic peft.parser.errors --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1

# 6. Publish graph sample từ loraplus.py vào Kafka
PYTHONPATH=src python3 -m group4_lab.cli parse-file \
  --file peft/src/peft/optimizers/loraplus.py \
  --repo-root peft \
  --repo-name huggingface/peft \
  --commit-sha 79f4c362248d3b3b4bc2ed24704ed3183528c53f \
  --publisher kafka \
  --bootstrap-servers localhost:9092

# 7. Apply Neo4j constraints và tạo Kafka Connector Sink
cat configs/neo4j/constraints.cypher | docker compose exec -T neo4j cypher-shell -u neo4j -p password123

curl -X POST http://localhost:8083/connectors \
  -H "Content-Type: application/json" \
  --data @configs/neo4j/node-sink-connector.json

curl -X POST http://localhost:8083/connectors \
  -H "Content-Type: application/json" \
  --data @configs/neo4j/edge-sink-connector.json

# 8. Publish full metadata 365 files vào Kafka
# File này được extract từ output/peft-events.jsonl trong lần chạy evidence.
docker compose exec -T kafka kafka-console-producer \
  --broker-list localhost:9092 \
  --topic peft.source.metadata \
  < evidence/full_metadata/peft.source.metadata.full.jsonl

# 9. Chạy Spark Structured Streaming ghi metadata vào MongoDB
JAVA_HOME=/tmp/group4-jdk17/Contents/Home \
GROUP4_SPARK_CHECKPOINT=checkpoints/full_metadata_stream \
GROUP4_MONGO_COLLECTION=peft_metadata_full \
spark-submit \
  --conf spark.jars.ivy=/tmp/group4-ivy \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.3,org.mongodb.spark:mongo-spark-connector_2.13:10.5.0 \
  jobs/mongo_streaming_available_now.py

# 10. Kiểm tra replay parser-level
PYTHONPATH=src python3 -m group4_lab.cli replay-file \
  --before-file evidence/replay/loraplus_before.py \
  --after-file evidence/replay/loraplus_after.py \
  --repo-root peft \
  --repo-name huggingface/peft \
  --commit-sha 79f4c362248d3b3b4bc2ed24704ed3183528c53f

# 11. Chạy tests
PYTHONPATH=src python3 -m pytest -q

# 12. Build Jupyter Book local
jupyter-book build docs/book
```


## 11. Reflection

Phần hoạt động ổn định nhất trong pipeline là parser, Kafka topic design và full metadata streaming sang MongoDB. Đây cũng là các phần có evidence end-to-end rõ nhất: parse summary, Kafka offsets, Spark input/output rows, MongoDB count và checkpoint state.

Phần khó nhất là Neo4j live ingestion. Full graph của 365 file sinh hơn 604k graph events, quá nặng để ingest live ổn định trong môi trường Docker/Neo4j local. Nhóm xử lý bằng cách parse full graph offline để chứng minh parser scalability, rồi kiểm chứng Neo4j live ingestion và replay/idempotency bằng file đại diện.